# VACE Control Video — WAN 2.1 — ComfyUI

WAN 2.1 VACE 14B ile video kontrol (pose, depth, edge).

## Colab Secrets
- `CF_TUNNEL_TOKEN` — Cloudflare tunnel
- `HF_TOKEN` — HuggingFace

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

# Bu workflow comfy-core ile çalışır, ekstra node gerekmez

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

MODELS_DIR = f'{COMFY_DIR}/models'

def hf_download(repo, filename, dest_dir):
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Ba\u015far\u0131s\u0131z: {e}')

# === Diffusion Model ===
print('\U0001f4e5 Diffusion Model:')
hf_download('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/diffusion_models/wan2.1_vace_14B_fp16.safetensors', f'{MODELS_DIR}/diffusion_models')

# === Text Encoder ===
print('\n\U0001f4e5 Text Encoder:')
hf_download('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/text_encoders/umt5_xxl_fp16.safetensors', f'{MODELS_DIR}/text_encoders')

# === VAE ===
print('\n\U0001f4e5 VAE:')
hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan_2.1_vae.safetensors', f'{MODELS_DIR}/vae')

# === LoRA (CausVid) ===
print('\n\U0001f4e5 LoRA:')
hf_download('Kijai/WanVideo_comfy', 'Wan21_CausVid_14B_T2V_lora_rank32.safetensors', f'{MODELS_DIR}/loras')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy
`USE_CLOUDFLARE = True`: Cloudflare tunnel

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False
PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI ba\u015flat\u0131ld\u0131 (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI \u00e7\u00f6kt\u00fc!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI haz\u0131r ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canl\u0131 tutma. Durdurmak i\u00e7in interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)